[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [3]:
# !echo $GT_TOKEN # makesure the token is loaded

In [4]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

Already up to date.


In [5]:
!pip install -r jrcai_corekit/requirements.txt

/bin/bash: /home/majed_alshaibani/Projects/instructions-tuning/venv/bin/pip: /home/majed_alshaibani/Projects/InstructionsTuning/venv/bin/python3: bad interpreter: No such file or directory


add jrcai_corekit to path

In [6]:
import sys
sys.path.append('jrcai_corekit/src')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Constants

In [8]:
TAWJEEH_DATASET_NAME = 'AraBench_dev'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/arabench_dev_experimental'
MODEL_PATH = "/hdd/shared_models/jais-13b"
TASK_NAME='dialect_identification'

In [9]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [10]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://tawjeeh.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14796,
  'tags': [],
  'name': 'Text Classification-Act.As.Expert',
  'task': {'name': 'text classification'},
  'status': 'SUBMITTED',
  'template': "Task: Imagine you're organizing content into categories. Based on the details in the text, classify it into one of the following categories: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %}, {% endif %}{% endfor %}.\r\n\r\nText: {{ text }}\r\n\r\nPlease respond only with the category that best describes this text.\r\n|||\r\n{{ answer_choices[label] }}",
  'dataset_name': 'arbml/ArCovidVac',
  'dataset_subset': 'default',
  'answer_choices': ['celebrity',
   'info_news',
   'personal',
   'unrelated',
   'plan',
   'requests',
   'others',
   'rumors',
   'advice',
   'restrictions'],
  'text_direction': 'ltr'},
 {'id': 14795,
  'tags': [],
  'name': 'Topic Classification-Act.As.Expert',
  'task': {'name': 'topic classification'},
  'status': 'SUBMITTED',
  'template': 'Task: Imagine you’re categorizing article

In [11]:
len(prompts)

264

In [12]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED', prompts))
len(filtered_prompts)

153

## Finetuning

### Get the dataset prompts

In [13]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(filtered_prompts)

153

### Download the dataset

In [14]:
import datasets

In [15]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['arabic', 'english', 'label'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['arabic', 'english', 'label'],
        num_rows: 10000
    })
})

### Merge the prompts

In [16]:
from jinja2 import Environment, StrictUndefined

In [17]:
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    prefix = prefix.strip()
    suffix = suffix.strip()
    punc = ['.', ':']
    for p in punc:
        if prefix.endswith(p):
            prefix = prefix[:-1]
    return f'{prefix.strip()} {suffix.strip()}'

In [18]:
def apply_template(prompt_template, sample):
    template = prompt_template['template']
    template = preprocess_template(template)
    sample['answer_choices'] = prompt_template['answer_choices']
    env = Environment(undefined=StrictUndefined)
    template = env.from_string(template)
    rendered_template = template.render(**sample)
    return rendered_template

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [19]:
example_prompt_template = dataset_prompts[1]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][2]))

Given the following Arabic text هيدا صالح ل اربعتاعشر يوم، فا فيك تستعملو لحد يوم تلاتة و عشرين., in what dialect was it written, choose from the following
Tunisian, MSA, Morrocan, Qatari, Egyptian, Lebanese Lebanese


In [20]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

15000.0

In [21]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
    
# rendered_train_prompts_dataset = list(
#     map(
#         lambda sample: apply_template(example_prompt_template, sample),
#         tqdm(hf_exp_dataset['train']),
#         )
#     )
len(rendered_train_prompts_dataset)

  0%|          | 0/30000 [00:00<?, ?it/s]

rending For the following Arabic text: {{arabic}}, the most probable dialect based on the spelling variations that represent dialectal pronunciation, among the following dialects: Tunisian, MSA, Moroccan, Qatari, Egyptian, Lebanese is: 
|||
{{answer_choices[label]}} sample index: 0
rending Given the following Arabic text {{arabic}}, in what dialect was it written, choose from the following
{{answer_choices | join(', ')}}
|||
{{answer_choices[label]}} sample index: 15000


30000

## Finetune the LLM

In [22]:
GLOBAL_SEED = 42

In [23]:
import random
random.seed(GLOBAL_SEED)

In [24]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, JAISInitializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [25]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=JAISInitializer(),
)
llm_loader

In [26]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/jais-13b/config.json
loading configuration file /hdd/shared_models/jais-13b/config.json
Model config JAISConfig {
  "_name_or_path": "/hdd/shared_models/jais-13b",
  "activation_function": "swiglu",
  "architectures": [
    "JAISLMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_jais.JAISConfig",
    "AutoModel": "modeling_jais.JAISModel",
    "AutoModelForCausalLM": "modeling_jais.JAISLMHeadModel",
    "AutoModelForQuestionAnswering": "modeling_jais.JAISForQuestionAnswering",
    "AutoModelForSequenceClassification": "modeling_jais.JAISForSequenceClassification",
    "AutoModelForTokenClassification": "modeling_jais.JAISForTokenClassification"
  },
  "bos_token_id": 0,
  "embd_pdrop": 0.0,
  "embeddings_scale": 14.6,
  "eos_token_id": 0,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "jais",
  "n_embd": 5120,
  "n_head": 40,
  "n_inner": 13653,
  "n_layer": 40,
  "n_position

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing JAISLMHeadModel.

All the weights of JAISLMHeadModel were initialized from the model checkpoint at /hdd/shared_models/jais-13b.
If your task is similar to the task the model of the checkpoint was trained on, you can already use JAISLMHeadModel for predictions without further training.
loading configuration file /hdd/shared_models/jais-13b/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 0,
  "eos_token_id": 0,
  "pad_token_id": 0
}

loading file tokenizer.json
loading file tokenizer.model
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading configuration file /hdd/shared_models/jais-13b/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 0,
  "eos_token_id": 0,
  "pad_token_id": 0
}



In [27]:
train_samples,eval_samples = train_test_split(rendered_train_prompts_dataset, test_size=0.1, random_state=GLOBAL_SEED)
def generate_tuple(sample):
    sample_words = sample.split(' ')
    prefix,suffix = ' '.join(sample_words[:-1]),f' {sample_words[-1]}'
    return prefix,suffix
train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(27000,
 3000,
 [('For the following Arabic text: ترتفع الاسعار الى اس الى الى ارقام قياسيه, the most probable dialect based on the spelling variations that represent dialectal pronunciation, among the following dialects: Tunisian, MSA, Moroccan, Qatari, Egyptian, Lebanese is',
   ' Qatari'),
  ('For the following Arabic text: عندي هيدا ., the most probable dialect based on the spelling variations that represent dialectal pronunciation, among the following dialects: Tunisian, MSA, Moroccan, Qatari, Egyptian, Lebanese is',
   ' Lebanese'),
  ('For the following Arabic text:  يرسل ٱبن ٱلإنسان ملائكته فيجمعون من ملكوته جميع ٱلمعاثر وفاعلي ٱلإثم،  , the most probable dialect based on the spelling variations that represent dialectal pronunciation, among the following dialects: Tunisian, MSA, Moroccan, Qatari, Egyptian, Lebanese is',
   ' MSA'),
  ('Given the following Arabic text انا اللي باط جبدي يمه, in what dialect was it written, choose from the following\nTunisian, MSA, Morrocan, Qatar

In [28]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.jais_v1(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:1150: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto hal

loading configuration file /hdd/shared_models/jais-13b/config.json
Model config JAISConfig {
  "_name_or_path": "inception-mbzuai/jais-13b",
  "activation_function": "swiglu",
  "architectures": [
    "JAISLMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_jais.JAISConfig",
    "AutoModel": "modeling_jais.JAISModel",
    "AutoModelForCausalLM": "modeling_jais.JAISLMHeadModel",
    "AutoModelForQuestionAnswering": "modeling_jais.JAISForQuestionAnswering",
    "AutoModelForSequenceClassification": "modeling_jais.JAISForSequenceClassification",
    "AutoModelForTokenClassification": "modeling_jais.JAISForTokenClassification"
  },
  "bos_token_id": 0,
  "embd_pdrop": 0.0,
  "embeddings_scale": 14.6,
  "eos_token_id": 0,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "jais",
  "n_embd": 5120,
  "n_head": 40,
  "n_inner": 13653,
  "n_layer": 40,
  "n_positions": 2048,
  "pad_token_id": 0,
  "position_embedding_type": "alibi",


{'eval_loss': 4.180125713348389, 'eval_model_preparation_time': 0.0004, 'eval_runtime': 85.1664, 'eval_samples_per_second': 35.225, 'eval_steps_per_second': 2.207}


***** Running training *****
  Num examples = 27,000
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 16,880
  Number of trainable parameters = 13,107,200


Step,Training Loss,Validation Loss,Model Preparation Time
1000,0.077600,0.064589,0.000400
2000,0.050300,0.058984,0.000400
3000,0.039800,0.055768,0.000400
4000,0.026000,0.052080,0.000400
5000,0.029300,0.048987,0.000400
6000,0.017900,0.055358,0.000400
7000,0.016100,0.067479,0.000400
8000,0.010100,0.077639,0.000400
9000,0.006400,0.078761,0.000400
10000,0.008800,0.075236,0.000400


...

{'eval_loss': 0.0787612795829773, 'eval_model_preparation_time': 0.0004, 'eval_runtime': 71.771, 'eval_samples_per_second': 41.8, 'eval_steps_per_second': 2.619, 'epoch': 5.331753554502369}



***** Running Evaluation *****
  Num examples = 3000
  Batch size = 16


{'eval_loss': 0.07523550093173981, 'eval_model_preparation_time': 0.0004, 'eval_runtime': 71.7478, 'eval_samples_per_second': 41.813, 'eval_steps_per_second': 2.62, 'epoch': 5.924170616113744}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.04898662492632866